# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library. 

### Dataset Source
The dataset is described by a Croissant schema available at the URL below.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Get dataset metadata as a Python dict
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}\n{metadata['description']}")

## 2. Data Overview
Review the available record sets, their fields, and associated `@id`s for referencing data elements.

In [ ]:
# List all record sets and their fields (referenced by @id)

record_sets = dataset.metadata.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs.id} | Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} | Name: {field.name} | Data Type: {getattr(field, 'data_type', 'unknown')}")
    print()

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. All referencing is performed using the record set and field `@id` values from the overview above.

In [ ]:
# Put the record set @id(s) you want to examine here
# (From section above, likely there is one main table with abundant protein quantifications.)

record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"--- Record set: {record_set_id} ---")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)

Demonstrate example EDA steps: filtering records by numeric field, normalizing a field, and grouping by a categorical field using precise `@id` references.

In [ ]:
# For this example, select the first record set and a representative numeric and group field

# Choose your target record set:
main_record_set_id = record_sets_ids[0]
df = dataframes[main_record_set_id]

# List available columns to help choose a numeric field
print(f"Columns in {main_record_set_id}:\n", df.columns.tolist())

# Choose numeric field (e.g., 'cr:coverage' might be coverage percentage)
# Make sure to select the ACTUAL column @id or field name as seen above.
# For demonstration, let's pick a generic numeric field, e.g., 'cr:coverage', and a grouping field 'cr:description' if present.
numeric_field_id = None
group_field_id = None
for col in df.columns:
    # Often, columns include 'coverage' or similar; you may update these filter strings as required.
    if numeric_field_id is None and ('coverage' in col.lower() or 'count' in col.lower() or 'abundance' in col.lower() or 'intensity' in col.lower()):
        numeric_field_id = col
    if group_field_id is None and ('description' in col.lower() or 'sample' in col.lower() or 'group' in col.lower()):
        group_field_id = col

if not numeric_field_id:
    raise ValueError("No numeric field found! Please check the df.columns and set numeric_field_id manually.")
else:
    print(f"Numeric field selected: {numeric_field_id}")
if not group_field_id:
    print("No obvious group field found. Grouping step will be skipped.")
else:
    print(f"Grouping field selected: {group_field_id}")

# Clean and convert the numeric field (sometimes numbers are strings)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter by threshold; choose e.g., 10 as arbitrary example
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records from '{main_record_set_id}' where {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally group
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization

Here we visualize the distribution of the selected numeric field and the group means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouped data available, plot means
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    grouped_df_sorted = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)  # Top 20 groups
    sns.barplot(data=grouped_df_sorted, x=numeric_field_id, y=group_field_id, palette="viridis")
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 20)")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- This notebook demonstrated loading a FAIR<sup>2</sup> Croissant dataset and exploring its protein quantification table by referencing all entities using their `@id`.
- You reviewed metadata, inspected available record sets and field structure, extracted and normalized a quantitative column, and visualized the main distributions and groups.
- To adapt this notebook to other FAIR<sup>2</sup> schemas, update the Croissant schema URL and adjust field/group names as appropriate to the relevant `@id` values.